# TP4 : tableau de bord d'indicateurs

## Objectifs

- construire et personnaliser des graphiques avec matplotlib : courbes, barres, histogrammes, nuages de points, grilles de graphiques
- ajouter des repères à un graphique : ligne d'objectif, labels de valeurs, annotations
- produire des graphiques statistiques avec seaborn
- créer des graphiques interactifs avec plotly et les exporter en HTML
- repérer et corriger des graphiques trompeurs ou mal construits
- assembler un tableau de bord qui applique les bonnes pratiques de dataviz

## Livrable

- ce notebook de restitution, avec un tableau de bord de 4 à 5 graphiques commentés, dont au moins un avec seaborn et un interactif avec plotly
- l'export HTML du graphique interactif principal, `ca_mensuel_regions.html`
- la figure `activite_mensuelle.png`

## Les données

Les deux tables produites au TP3 :

- `indicateurs_mois_region.parquet` : une ligne par mois et par région
- `ventes_consolidees.parquet` : une ligne par commande

Une version est fournie dans `ressources/`. Tu peux utiliser les tiennes à la place en modifiant la variable `DOSSIER`.

## Comment travailler

- Le TP suit le cours : chaque partie se fait juste après la section du cours correspondante.
- Complète les cellules marquées `# TODO`, en utilisant les noms de variables indiqués dans l'énoncé.
- Quand un **résultat attendu** est indiqué, compare-le avec ce que tu obtiens. Pour un graphique, le résultat attendu est une image : ton graphique doit lui ressembler, sans être identique au pixel près.
- Les cellules qui commencent par `# Vérification` contrôlent ton travail avec `assert` : exécute-les sans les modifier. Si tout est juste, elles affichent `OK` ; sinon, elles s'arrêtent sur une `AssertionError` dont le message indique ce qui ne va pas. Pour les graphiques, elles inspectent la figure `fig` et le graphique `ax` (ou `axes`) : garde ces noms.
- Les questions **Pour aller plus loin** sont facultatives : fais-les si tu as terminé une partie en avance.

## Partie 0 : environnement et données

*À faire après la section 1 du cours : les données et l'environnement.*

Active l'environnement virtuel des TP précédents, ou recrée-en un dans ce dossier avec `python3 -m venv .venv`, puis `python -m pip install -r ../tp01-environnement-pandas/requirements.txt`. Ajoute ensuite les bibliothèques de visualisation et mets à jour le `requirements.txt` :

```bash
python -m pip install matplotlib seaborn plotly
python -m pip freeze > requirements.txt
```

Lance `jupyter lab` depuis le dossier de ce TP, puis exécute la cellule suivante : elle importe les bibliothèques et charge les deux tables.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import plotly.express as px
import seaborn as sns
from matplotlib.ticker import FuncFormatter

DOSSIER = Path("ressources")   # ou Path("../tp03-indicateurs") pour utiliser tes propres fichiers

indicateurs = pd.read_parquet(DOSSIER / "indicateurs_mois_region.parquet")
ventes = pd.read_parquet(DOSSIER / "ventes_consolidees.parquet")
indicateurs.head()


### Question 0.1

Affiche les dimensions des deux tables, le premier et le dernier mois de `indicateurs` (`min` et `max`), puis la liste triée des régions (`unique`, puis `sorted`).

**Résultat attendu** :

```text
(54, 8) (2182, 15)
2025-01-01 00:00:00 2025-09-01 00:00:00
['Auvergne-Rhône-Alpes', 'Bretagne', 'Hauts-de-France', 'Nouvelle-Aquitaine', 'Occitanie', 'Île-de-France']
```

In [ ]:
# TODO


### Question 0.2

Les graphiques de ce TP reviennent souvent aux mêmes agrégats. Calcule :

- `mensuel` : le chiffre d'affaires total de chaque mois, toutes régions confondues ;
- `par_region` : le chiffre d'affaires total de chaque région sur les neuf mois, trié par ordre croissant.

Affiche-les.

**Résultat attendu** :

```text
mois
2025-01-01    43475.13
2025-02-01    37830.25
2025-03-01    53471.97
2025-04-01    56339.59
2025-05-01    48801.55
2025-06-01    52362.65
2025-07-01    31873.02
2025-08-01    27273.63
2025-09-01    58073.80
Name: ca, dtype: float64
region
Bretagne                 41079.71
Occitanie                50318.18
Hauts-de-France          53467.33
Auvergne-Rhône-Alpes     64856.25
Nouvelle-Aquitaine       71843.18
Île-de-France           127936.94
Name: ca, dtype: float64
```

In [ ]:
# TODO


In [ ]:
# Vérification
assert len(mensuel) == 9, "mensuel doit contenir une valeur par mois, soit 9 valeurs"
assert round(mensuel.sum(), 2) == round(indicateurs["ca"].sum(), 2), "le total de mensuel doit être égal au chiffre d'affaires total"
assert len(par_region) == 6, "par_region doit contenir les 6 régions"
assert list(par_region) == sorted(par_region), "par_region doit être trié par ordre croissant"
print("Question 0.2 : OK")


## Partie 1 : matplotlib

*À faire après la section 2 du cours : matplotlib.*

Chaque graphique se crée avec `fig, ax = plt.subplots(figsize=...)`, en reprenant la taille indiquée pour obtenir un résultat comparable à l'image attendue.

### Question 1.1

Trace l'évolution du chiffre d'affaires mensuel (`mensuel`) avec une courbe et des marqueurs ronds, dans `fig, ax = plt.subplots(figsize=(8, 4))`. Ajoute :

- un titre, et un nom à l'axe vertical avec son unité ;
- un axe vertical qui part de zéro ;
- des montants affichés en milliers d'euros, grâce à un formateur `FuncFormatter` rangé dans une variable `milliers`, que tu réutiliseras dans la suite du TP ;
- une grille horizontale discrète.

**Résultat attendu** :

![Résultat attendu de la question 1.1](attendus/q1_1.png)

In [ ]:
# TODO


In [ ]:
# Vérification
assert ax.get_title() != "", "le graphique doit avoir un titre"
assert ax.get_ylabel() != "", "l'axe vertical doit être nommé, avec son unité"
assert len(ax.lines) == 1 and len(ax.lines[0].get_xdata()) == 9, "il faut une seule courbe, avec un point par mois"
assert ax.get_ylim()[0] == 0, "l'axe vertical doit partir de zéro : ax.set_ylim(0, ...)"
assert ax.get_ylim()[1] >= mensuel.max(), "l'axe vertical doit monter au moins jusqu'au meilleur mois"
assert "k" in ax.yaxis.get_major_formatter()(20000, 0), "les montants doivent être affichés en milliers, par exemple 20 k€"
print("Question 1.1 : OK")


### Question 1.2

Trace maintenant le chiffre d'affaires mensuel en barres verticales, dans `fig, ax = plt.subplots(figsize=(8, 4))`. Pour que les barres aient une largeur normale, place-les sur des labels texte : `etiquettes = mensuel.index.strftime("%m/%y")`.

Calcule ensuite dans `objectif_total` l'objectif mensuel de l'ensemble des régions, c'est-à-dire la somme des objectifs mensuels des six régions pour un mois. Trace-le comme une ligne horizontale en pointillés (`ax.axhline`), donne un `label` aux barres et à la ligne, et affiche la légende en haut à gauche, sur deux colonnes, en laissant de la place au-dessus des barres (axe vertical jusqu'à 70 000 euros). Réutilise le formateur `milliers`.

Affiche enfin la valeur de `objectif_total` et la liste des mois qui dépassent l'objectif. Combien y en a-t-il ?

Indice : `indicateurs.groupby("mois")["objectif_mensuel"].sum()` donne l'objectif total de chaque mois.

**Résultat attendu** :

![Résultat attendu de la question 1.2](attendus/q1_2.png)

In [ ]:
# TODO


In [ ]:
# Vérification
assert len(ax.patches) == 9, "il faut une barre par mois, soit 9 barres"
assert ax.get_ylim()[0] == 0, "l'axe vertical d'un graphique en barres doit partir de zéro"
assert objectif_total == 47000, "l'objectif total est la somme des objectifs mensuels des six régions"
assert any(set(ligne.get_ydata()) == {objectif_total} for ligne in ax.lines), "trace l'objectif avec ax.axhline(objectif_total, ...)"
assert ax.get_legend() is not None, "affiche la légende avec ax.legend()"
print("Question 1.2 : OK")


_Ta réponse ici._

### Question 1.3

Trace le chiffre d'affaires total par région (`par_region`) en barres horizontales, dans `fig, ax = plt.subplots(figsize=(8, 4))`, avec la plus grande barre en haut.

- Mets en évidence la région la plus forte avec une couleur, et les autres en gris : construis la liste `couleurs` avec une compréhension et `par_region.idxmax()`.
- Écris un titre qui énonce le message du graphique, et nomme l'axe horizontal.
- Affiche au bout de chaque barre son montant en milliers d'euros avec `ax.bar_label`, et prolonge l'axe horizontal jusqu'à 145 000 euros (`ax.set_xlim`) pour laisser de la place aux labels.

**Résultat attendu** :

![Résultat attendu de la question 1.3](attendus/q1_3.png)

In [ ]:
# TODO


In [ ]:
# Vérification
largeurs = [barre.get_width() for barre in ax.patches]
assert len(largeurs) == 6, "il faut une barre par région"
assert largeurs == sorted(largeurs), "la plus grande barre doit être en haut : trace par_region, trié par ordre croissant"
assert len({barre.get_facecolor() for barre in ax.patches}) == 2, "deux couleurs seulement : une pour la région la plus forte, du gris pour les autres"
assert ax.patches[-1].get_facecolor() != ax.patches[0].get_facecolor(), "c'est la région la plus forte, en haut, qui doit être mise en évidence"
assert len(ax.texts) == 6, "ajoute un label par barre avec ax.bar_label"
assert ax.get_title() != "" and ax.get_xlabel() != "", "il faut un titre et un axe horizontal nommé"
print("Question 1.3 : OK")


### Question 1.4

Calcule dans `commandes` le nombre total de commandes de chaque mois. Dans une même figure, `fig, axes = plt.subplots(1, 2, figsize=(11, 4))`, place côte à côte le chiffre d'affaires mensuel et le nombre de commandes mensuel, en barres, chacun avec un titre. Utilise `fig.tight_layout()`, puis enregistre la figure en `activite_mensuelle.png`.

Le creux de l'été vient-il d'une baisse du nombre de commandes, ou d'une baisse du montant de chaque commande ?

**Résultat attendu** :

![Résultat attendu de la question 1.4](attendus/q1_4.png)

In [ ]:
# TODO


In [ ]:
# Vérification
assert commandes.sum() == indicateurs["nb_commandes"].sum(), "commandes doit contenir le nombre total de commandes de chaque mois"
assert len(fig.axes) == 2, "la figure doit contenir deux graphiques côte à côte"
assert [len(graphique.patches) for graphique in fig.axes] == [9, 9], "chaque graphique doit avoir une barre par mois"
assert all(graphique.get_title() != "" for graphique in fig.axes), "chaque graphique doit avoir un titre"
assert Path("activite_mensuelle.png").exists(), "enregistre la figure avec fig.savefig"
print("Question 1.4 : OK")


_Ta réponse ici._

### Question 1.5

Trace avec `ax.hist` l'histogramme du taux de réalisation de l'objectif (`taux_realisation`) des 54 couples mois et région, dans `fig, ax = plt.subplots(figsize=(8, 4))`. Utilise des classes de 10 points, de 20 % à 180 % : passe à `bins` la liste des bornes, construite avec une compréhension, `[x / 10 for x in range(2, 19)]`.

Ajoute une ligne verticale à 100 % (`ax.axvline`) avec une légende, nomme les deux axes et affiche les taux en pourcentage avec un formateur. Combien de couples mois et région atteignent leur objectif ?

**Résultat attendu** :

![Résultat attendu de la question 1.5](attendus/q1_5.png)

In [ ]:
# TODO


In [ ]:
# Vérification
assert len(ax.patches) == 16, "il faut 16 classes de 10 points, de 20 % à 180 %"
assert sum(barre.get_height() for barre in ax.patches) == 54, "l'histogramme doit compter les 54 couples mois et région"
assert any(set(ligne.get_xdata()) == {1} for ligne in ax.lines), "trace la ligne de l'objectif avec ax.axvline(1, ...)"
assert ax.get_xlabel() != "" and ax.get_ylabel() != "", "nomme les deux axes"
print("Question 1.5 : OK")


_Ta réponse ici._

### Question 1.6

Trace avec `ax.scatter` le chiffre d'affaires de chaque couple mois et région en fonction de son nombre de commandes, dans `fig, ax = plt.subplots(figsize=(8, 4))`. Rends les points semi-transparents (`alpha=0.7`), nomme les axes et donne un titre. Les deux grandeurs sont-elles liées ?

**Résultat attendu** :

![Résultat attendu de la question 1.6](attendus/q1_6.png)

In [ ]:
# TODO


In [ ]:
# Vérification
assert len(ax.collections) == 1 and len(ax.collections[0].get_offsets()) == 54, "il faut un point par couple mois et région, soit 54 points"
assert ax.get_xlabel() != "" and ax.get_ylabel() != "", "nomme les deux axes"
assert ax.get_title() != "", "le graphique doit avoir un titre"
print("Question 1.6 : OK")


_Ta réponse ici._

### Question 1.7

Construis dans `ca_categories` un tableau croisé du chiffre d'affaires de `ventes`, avec les mois en lignes, au format `%m/%y`, et les catégories en colonnes : `pivot_table` avec `index=ventes["date"].dt.strftime("%m/%y")`, `columns="categorie"`, `values="montant"` et `aggfunc="sum"`.

Trace ensuite, dans une grille de 2 lignes et 2 colonnes qui partagent la même échelle verticale, `fig, axes = plt.subplots(2, 2, figsize=(10, 6), sharey=True)`, un graphique en barres par catégorie pour les quatre catégories `Informatique`, `Bureau`, `Papeterie` et `Électroménager`, chacun avec le nom de sa catégorie en titre. Donne un titre à la figure avec `fig.suptitle`.

Pourquoi `sharey=True` est-il indispensable ici ?

Indice : une boucle `for ax, categorie in zip(axes.flat, categories):` parcourt les quatre graphiques de la grille en même temps que les catégories.

**Résultat attendu** :

![Résultat attendu de la question 1.7](attendus/q1_7.png)

In [ ]:
# TODO


In [ ]:
# Vérification
assert ca_categories.shape == (9, 5), "ca_categories doit avoir 9 mois en lignes et 5 catégories en colonnes"
assert round(ca_categories.sum().sum(), 2) == round(ventes["montant"].sum(), 2), "le tableau doit contenir tout le chiffre d'affaires"
assert len(fig.axes) == 4, "la grille doit contenir 4 graphiques"
assert len({graphique.get_ylim() for graphique in fig.axes}) == 1, "les 4 graphiques doivent partager la même échelle verticale : sharey=True"
assert all(graphique.get_title() != "" for graphique in fig.axes), "chaque graphique doit avoir le nom de sa catégorie en titre"
print("Question 1.7 : OK")


_Ta réponse ici._

### Pour aller plus loin 1.8

Reprends la courbe de la question 1.1 et signale le creux de l'été : colore la période de mi-juin à mi-août avec `ax.axvspan` (bornes `pd.to_datetime("2025-06-15")` et `pd.to_datetime("2025-08-15")`), et ajoute une annotation avec une flèche (`ax.annotate`) qui pointe sur le mois d'août. Change le titre pour qu'il énonce le message.

**Résultat attendu** :

![Résultat attendu de la question 1.8](attendus/q1_8.png)

In [ ]:
# TODO


In [ ]:
# Vérification
assert len(ax.patches) >= 1, "colore la période de l'été avec ax.axvspan"
assert len(ax.texts) == 1, "ajoute une annotation avec ax.annotate"
print("Question 1.8 : OK")


### Pour aller plus loin 1.9

Avec le raccourci de pandas, trace `par_region` en barres horizontales en une seule ligne, dans un `ax` créé avec `fig, ax = plt.subplots(figsize=(8, 4))` : `par_region.plot.barh(ax=ax)`. Ajoute un titre et nomme l'axe horizontal. Que met pandas sur l'axe vertical ?

In [ ]:
# TODO


In [ ]:
# Vérification
assert len(ax.patches) == 6, "il faut une barre par région"
print("Question 1.9 : OK")


_Ta réponse ici._

## Partie 2 : seaborn

*À faire après la section 3 du cours : seaborn.*

Exécute la cellule suivante : elle applique un style sobre à tous les graphiques qui suivent, y compris ceux faits avec matplotlib.

In [ ]:
sns.set_theme(style="whitegrid")


### Question 2.1

Calcule la moyenne et la médiane du montant des commandes, arrondies à 2 décimales, dans les variables `moyenne` et `mediane`, et affiche-les.

Trace ensuite l'histogramme du montant des commandes avec `sns.histplot` et 40 classes, dans `fig, ax = plt.subplots(figsize=(8, 4))`. Ajoute deux lignes verticales, à la moyenne et à la médiane, avec une légende. Comment qualifierais-tu cette distribution ? Pourquoi la moyenne est-elle si éloignée de la médiane ?

**Résultat attendu** :

![Résultat attendu de la question 2.1](attendus/q2_1.png)

In [ ]:
# TODO


In [ ]:
# Vérification
assert moyenne == 187.67 and mediane == 104.7, "calcule la moyenne et la médiane des montants, arrondies à 2 décimales"
assert len(ax.patches) == 40, "l'histogramme doit avoir 40 classes : bins=40"
assert len(ax.lines) >= 2, "ajoute les deux lignes verticales avec ax.axvline"
assert ax.get_title() != "" and ax.get_xlabel() != "", "il faut un titre et un axe horizontal nommé"
print("Question 2.1 : OK")


_Ta réponse ici._

### Question 2.2

Compare la distribution du montant des commandes entre catégories avec une boîte à moustaches, dans `fig, ax = plt.subplots(figsize=(8, 4))`. Range dans `ordre` les catégories triées par médiane décroissante (`groupby`, `median`, `sort_values`, puis `.index`), et passe-le au paramètre `order` de `sns.boxplot`.

Affiche ensuite les commandes de plus de 1 000 euros, comptées par catégorie, produit et quantité (`groupby` sur ces trois colonnes, puis `size()`).

1. Quelle catégorie a les valeurs extrêmes les plus élevées ? À quoi correspondent-elles ?
2. La papeterie a aussi beaucoup de points au-delà de ses moustaches, alors que ses commandes sont modestes. Pourquoi ?

**Résultat attendu** :

![Résultat attendu de la question 2.2](attendus/q2_2.png)

In [ ]:
# TODO


In [ ]:
# Vérification
assert list(ordre) == ["Non référencé", "Informatique", "Électroménager", "Bureau", "Papeterie"], "les catégories doivent être triées par médiane décroissante"
assert [etiquette.get_text() for etiquette in ax.get_xticklabels()] == list(ordre), "passe ordre au paramètre order de sns.boxplot"
print("Question 2.2 : OK")


_Ta réponse ici._

### Question 2.3

Avec `sns.countplot`, trace le nombre de commandes par catégorie, dans `fig, ax = plt.subplots(figsize=(8, 4))`, de la catégorie la plus fréquente à la moins fréquente : passe au paramètre `order` l'index de `ventes["categorie"].value_counts()`.

**Résultat attendu** :

![Résultat attendu de la question 2.3](attendus/q2_3.png)

In [ ]:
# TODO


In [ ]:
# Vérification
hauteurs = [barre.get_height() for barre in ax.patches]
assert len(hauteurs) == 5, "il faut une barre par catégorie"
assert sum(hauteurs) == len(ventes), "countplot compte une ligne par commande"
assert hauteurs == sorted(hauteurs, reverse=True), "les barres doivent aller de la plus haute à la plus basse : paramètre order"
print("Question 2.3 : OK")


### Question 2.4

Trace maintenant le chiffre d'affaires par catégorie avec `sns.barplot`, dans `fig, ax = plt.subplots(figsize=(8, 4))` : la somme des montants (`estimator="sum"`), sans trait d'incertitude (`errorbar=None`), et dans le même ordre qu'à la question précédente.

Compare avec le nombre de commandes : que remarques-tu pour la papeterie et pour le bureau ?

**Résultat attendu** :

![Résultat attendu de la question 2.4](attendus/q2_4.png)

In [ ]:
# TODO


In [ ]:
# Vérification
hauteurs = [barre.get_height() for barre in ax.patches]
assert len(hauteurs) == 5, "il faut une barre par catégorie"
assert round(sum(hauteurs)) == round(ventes["montant"].sum()), "les barres doivent représenter des totaux : estimator=\"sum\""
assert [etiquette.get_text() for etiquette in ax.get_xticklabels()] == list(ventes["categorie"].value_counts().index), "garde l'ordre de la question précédente"
print("Question 2.4 : OK")


_Ta réponse ici._

### Question 2.5

Avec `sns.lineplot`, trace l'évolution du chiffre d'affaires mensuel de chaque région sur un même graphique (`hue="region"`), avec des marqueurs ronds, dans `fig, ax = plt.subplots(figsize=(8, 4))`. Pour qu'elle ne cache pas les courbes, place la légende à droite du graphique : `sns.move_legend(ax, "upper left", bbox_to_anchor=(1, 1))`. Ce graphique est-il facile à lire ?

**Résultat attendu** :

![Résultat attendu de la question 2.5](attendus/q2_5.png)

In [ ]:
# TODO


In [ ]:
# Vérification
regions = {texte.get_text() for texte in ax.get_legend().get_texts()}
assert regions == set(indicateurs["region"]), "il faut une courbe par région, avec sa légende : hue=\"region\""
print("Question 2.5 : OK")


_Ta réponse ici._

### Question 2.6

Avec `sns.relplot`, trace l'évolution du chiffre d'affaires mensuel avec un graphique par région (`col="region"`, `col_wrap=3`, `kind="line"`), dans une variable `g`. Pour une figure compacte, utilise `height=2.8` et `aspect=1.4`, puis titre chaque graphique avec le seul nom de sa région (`g.set_titles("{col_name}")`).

Pourquoi est-ce plus lisible que les six courbes de la question précédente ? Les échelles sont-elles identiques d'un graphique à l'autre ?

**Résultat attendu** :

![Résultat attendu de la question 2.6](attendus/q2_6.png)

In [ ]:
# TODO


In [ ]:
# Vérification
assert len(g.axes.flat) == 6, "il faut un graphique par région"
assert len({graphique.get_ylim() for graphique in g.axes.flat}) == 1, "les graphiques doivent partager la même échelle verticale"
print("Question 2.6 : OK")


_Ta réponse ici._

### Question 2.7

Construis dans `taux` une heatmap du taux de réalisation de l'objectif, avec les régions en lignes et les mois en colonnes, au format `%m/%y` : `pivot_table` avec `columns=indicateurs["mois"].dt.strftime("%m/%y")`. Trace-la dans `fig, ax = plt.subplots(figsize=(9, 4))`. Affiche les valeurs en pourcentage et centre la palette sur 100 %, avec une palette lisible par les daltoniens. La barre de couleur est inutile quand les valeurs sont écrites : retire-la avec `cbar=False`.

Quelle région atteint le plus souvent son objectif ? Que se passe-t-il en juillet et en août ?

**Résultat attendu** :

![Résultat attendu de la question 2.7](attendus/q2_7.png)

In [ ]:
# TODO


In [ ]:
# Vérification
assert taux.shape == (6, 9), "taux doit avoir les 6 régions en lignes et les 9 mois en colonnes"
assert len(ax.texts) == 54, "écris la valeur de chaque case avec annot=True"
assert "Gn" not in ax.collections[0].get_cmap().name, "évite les palettes qui opposent le rouge et le vert"
print("Question 2.7 : OK")


_Ta réponse ici._

### Pour aller plus loin 2.8

Calcule dans `correlations` la matrice des corrélations entre les colonnes `ca`, `nb_commandes`, `panier_moyen`, `quantite` et `taux_realisation` d'`indicateurs`, avec la méthode `corr`. Représente-la dans `fig, ax = plt.subplots(figsize=(6, 4.5))` avec une heatmap annotée, avec deux décimales (`fmt=".2f"`), et une palette divergente qui va de -1 à 1 (`vmin=-1`, `vmax=1`).

Quelle grandeur est la plus liée au chiffre d'affaires ? Le panier moyen est-il lié au nombre de commandes ?

**Résultat attendu** :

![Résultat attendu de la question 2.8](attendus/q2_8.png)

In [ ]:
# TODO


In [ ]:
# Vérification
assert correlations.shape == (5, 5), "la matrice doit croiser les 5 colonnes demandées"
assert round(correlations.loc["ca", "nb_commandes"], 2) == 0.92, "calcule la matrice avec la méthode corr"
assert len(ax.texts) == 25, "écris la valeur de chaque case avec annot=True"
print("Question 2.8 : OK")


_Ta réponse ici._

### Pour aller plus loin 2.9

Les clients fidèles passent-ils de plus grosses commandes ? Compare les deux distributions du montant avec `sns.histplot` et `hue="client_fidele"`, dans `fig, ax = plt.subplots(figsize=(8, 4))`. Les deux groupes n'ont pas la même taille : pour les comparer, trace des densités (`stat="density"`) normalisées séparément (`common_norm=False`). Confirme avec la médiane et la moyenne de chaque groupe.

**Résultat attendu** :

![Résultat attendu de la question 2.9](attendus/q2_9.png)

In [ ]:
# TODO


_Ta réponse ici._

### Pour aller plus loin 2.10

Trace `sns.barplot(data=ventes, x="region", y="montant", ax=ax)`, sans préciser `estimator`, dans `fig, ax = plt.subplots(figsize=(8, 4))`. Que représentent la hauteur des barres et les traits verticaux ? Retrouve la hauteur des barres avec un `groupby`.

In [ ]:
# TODO


_Ta réponse ici._

## Partie 3 : plotly

*À faire après la section 4 du cours : plotly.*

Les graphiques plotly sont interactifs : survole-les, zoome, clique sur la légende. Les vérifications inspectent l'objet renvoyé par plotly express, que tu ranges dans la variable indiquée.

### Question 3.1

Crée avec `px.line` un graphique interactif du chiffre d'affaires mensuel par région, avec des marqueurs et un titre, dans une variable `fig_ca`. Ajoute dans l'info-bulle le nombre de commandes et le taux de réalisation formaté en pourcentage (`hover_data`), et remplace les noms de colonnes par des libellés lisibles avec `labels`. Termine par `fig_ca.show()`.

In [ ]:
# TODO


In [ ]:
# Vérification
assert len(fig_ca.data) == 6, "il faut une courbe par région : color=\"region\""
assert fig_ca.layout.title.text, "ajoute un titre avec title=..."
assert fig_ca.data[0].customdata is not None and fig_ca.data[0].customdata.shape[1] >= 2, "ajoute le nombre de commandes et le taux de réalisation à l'info-bulle avec hover_data"
info_bulle = fig_ca.data[0].hovertemplate
assert "region=" not in info_bulle and "ca=" not in info_bulle, "remplace les noms de colonnes par des libellés lisibles avec labels"
print("Question 3.1 : OK")


### Question 3.2

Avec `px.bar`, trace le chiffre d'affaires mensuel en barres empilées, une couleur par région (`color="region"`), avec un titre et des libellés lisibles, dans une variable `fig_barres`.

Survole les barres, puis clique sur une région dans la légende pour la masquer : que deviennent les barres ? Qu'apporte ce graphique par rapport à celui de la question 3.1 ?

In [ ]:
# TODO


In [ ]:
# Vérification
assert len(fig_barres.data) == 6, "il faut une couleur par région : color=\"region\""
assert all(trace.type == "bar" for trace in fig_barres.data), "utilise px.bar"
assert fig_barres.layout.title.text, "ajoute un titre"
print("Question 3.2 : OK")


_Ta réponse ici._

### Question 3.3

Avec `px.box`, trace la distribution du montant des commandes par catégorie, avec un titre, dans une variable `fig_boites`. Ajoute le produit et la quantité à l'info-bulle (`hover_data=["produit", "quantite"]`). Survole les points les plus hauts de la catégorie Bureau : retrouves-tu les commandes identifiées à la question 2.2 ?

In [ ]:
# TODO


In [ ]:
# Vérification
assert fig_boites.data[0].type == "box", "utilise px.box"
assert fig_boites.data[0].customdata is not None, "ajoute le produit et la quantité à l'info-bulle avec hover_data"
print("Question 3.3 : OK")


_Ta réponse ici._

### Question 3.4

Crée un nuage de points interactif du chiffre d'affaires en fonction du nombre de commandes, avec une couleur par région et une taille de point selon la quantité vendue (`size="quantite"`), dans une variable `fig_nuage`. Ajoute le mois à l'info-bulle, au format mois et année (`hover_data={"mois": "|%m/%Y"}`), des libellés lisibles et un titre qui énonce le message.

Survole les points en haut à droite, ceux qui s'écartaient de la tendance à la question 1.6 : à quelle région et à quels mois correspondent-ils ?

In [ ]:
# TODO


In [ ]:
# Vérification
assert len(fig_nuage.data) == 6, "il faut une couleur par région : color=\"region\""
assert fig_nuage.data[0].marker.size is not None, "la taille des points doit dépendre de la quantité : size=\"quantite\""
assert fig_nuage.layout.title.text, "ajoute un titre"
print("Question 3.4 : OK")


_Ta réponse ici._

### Question 3.5

Exporte le graphique de la question 3.1 en `ca_mensuel_regions.html` avec les options par défaut, puis en `ca_mensuel_regions_cdn.html` avec `include_plotlyjs="cdn"`. Affiche la taille des deux fichiers en kilo-octets (`Path(...).stat().st_size` donne la taille en octets), et ouvre-les dans ton navigateur. Lequel choisirais-tu pour l'envoyer par mail ?

In [ ]:
# TODO


In [ ]:
# Vérification
assert Path("ca_mensuel_regions.html").exists(), "exporte fig_ca en ca_mensuel_regions.html"
assert Path("ca_mensuel_regions_cdn.html").exists(), "exporte fig_ca en ca_mensuel_regions_cdn.html, avec include_plotlyjs=\"cdn\""
assert Path("ca_mensuel_regions.html").stat().st_size > 1_000_000, "le fichier par défaut embarque la bibliothèque plotly : il doit peser plusieurs Mo"
assert Path("ca_mensuel_regions_cdn.html").stat().st_size < 100_000, "avec include_plotlyjs=\"cdn\", le fichier doit peser moins de 100 Ko"
print("Question 3.5 : OK")


_Ta réponse ici._

### Pour aller plus loin 3.6

Avec `px.histogram`, trace le nombre de commandes par jour de la semaine (colonne `jour`), dans une variable `fig_jours`. Les jours doivent apparaître dans l'ordre du calendrier, du lundi au dimanche : passe la liste des jours dans cet ordre au paramètre `category_orders={"jour": jours}`. Pourquoi ne pas trier ces barres par valeur, comme on l'a fait pour les régions ?

In [ ]:
# TODO


In [ ]:
# Vérification
assert list(fig_jours.layout.xaxis.categoryarray) == jours, "les jours doivent suivre l'ordre du calendrier : category_orders"
print("Question 3.6 : OK")


_Ta réponse ici._

## Partie 4 : corriger des graphiques trompeurs

*À faire après la section 5 du cours : bonnes pratiques de dataviz.*

Chaque cellule de cette partie contient le code d'un graphique mal construit. Pour chacun :

1. exécute la cellule telle quelle et observe le graphique ;
2. explique le défaut dans la cellule de réponse : en quoi le graphique trompe-t-il ou gêne-t-il la lecture ?
3. modifie le code de la cellule pour obtenir le résultat attendu, en gardant les noms `fig` et `ax` (ou `axes`), puis exécute la vérification.

### Question 4.1

Le graphique suivant compare le panier moyen des régions. Corrige-le pour qu'il ne trompe plus, et donne-lui un titre qui énonce le message.

**Résultat attendu** :

![Résultat attendu de la question 4.1](attendus/q4_1.png)

In [ ]:
panier_region = ventes.groupby("region")["montant"].mean().sort_values()

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(panier_region.index, panier_region.values, color="#1F3A5F")
ax.set_ylim(150, 235)
ax.set_title("Panier moyen par région")
ax.set_ylabel("panier moyen (euros)")
ax.tick_params(axis="x", labelrotation=20)


In [ ]:
# Vérification
assert len(ax.patches) == 6, "il faut une barre par région"
assert ax.get_ylim()[0] == 0, "l'axe d'un graphique en barres doit partir de zéro"
print("Question 4.1 : OK")


_Ta réponse ici._

### Question 4.2

Le graphique suivant doit montrer quels produits sont les plus commandés. Corrige-le pour qu'on les trouve d'un coup d'œil, et donne-lui un titre qui énonce le message.

**Résultat attendu** :

![Résultat attendu de la question 4.2](attendus/q4_2.png)

In [ ]:
commandes_produit = ventes["produit"].value_counts().sort_index()

fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(commandes_produit.index, commandes_produit.values, color="#1F3A5F")
ax.set_title("Nombre de commandes par produit")
ax.set_xlabel("nombre de commandes")


In [ ]:
# Vérification
largeurs = [barre.get_width() for barre in ax.patches]
assert len(largeurs) == 17, "il faut une barre par produit"
assert largeurs == sorted(largeurs), "trie les barres pour que le produit le plus commandé soit en haut"
print("Question 4.2 : OK")


_Ta réponse ici._

### Question 4.3

Le graphique suivant présente le chiffre d'affaires par catégorie. Corrige ses couleurs : une seule couleur pour mettre en évidence la catégorie qui fait le plus de chiffre d'affaires, du gris pour les autres, et pas de légende. Donne-lui un titre qui énonce le message.

**Résultat attendu** :

![Résultat attendu de la question 4.3](attendus/q4_3.png)

In [ ]:
ca_categorie = ventes.groupby("categorie")["montant"].sum().sort_values()

fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(
    ca_categorie.index,
    ca_categorie.values,
    color=["red", "orange", "gold", "green", "purple"],
    label=list(ca_categorie.index),
)
ax.legend()
ax.set_title("Chiffre d'affaires par catégorie")
ax.set_xlabel("chiffre d'affaires (euros)")
ax.xaxis.set_major_formatter(milliers)


In [ ]:
# Vérification
assert len(ax.patches) == 5, "il faut une barre par catégorie"
assert len({barre.get_facecolor() for barre in ax.patches}) <= 2, "deux couleurs au plus : une pour mettre en évidence, du gris pour le reste"
assert ax.get_legend() is None, "la légende répète les libellés de l'axe : supprime-la"
print("Question 4.3 : OK")


_Ta réponse ici._

### Question 4.4

Les deux graphiques suivants comparent l'activité de l'Île-de-France et de la Bretagne. Corrige-les pour que la comparaison soit juste : la même échelle verticale pour les deux, qui part de zéro.

**Résultat attendu** :

![Résultat attendu de la question 4.4](attendus/q4_4.png)

In [ ]:
idf = indicateurs[indicateurs["region"] == "Île-de-France"]
bretagne = indicateurs[indicateurs["region"] == "Bretagne"]

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].plot(etiquettes, idf["ca"], marker="o", color="#1F3A5F")
axes[0].set_title("Île-de-France")
axes[0].yaxis.set_major_formatter(milliers)
axes[1].plot(etiquettes, bretagne["ca"], marker="o", color="#1F3A5F")
axes[1].set_title("Bretagne")
axes[1].yaxis.set_major_formatter(milliers)
fig.suptitle("Chiffre d'affaires mensuel (euros)")
fig.tight_layout()


In [ ]:
# Vérification
assert len(fig.axes) == 2, "garde les deux graphiques côte à côte"
assert axes[0].get_ylim() == axes[1].get_ylim(), "les deux graphiques doivent avoir la même échelle verticale : sharey=True"
assert axes[0].get_ylim()[0] == 0, "l'échelle verticale doit partir de zéro"
print("Question 4.4 : OK")


_Ta réponse ici._

### Question 4.5

Le graphique suivant montre la part de chaque produit dans le chiffre d'affaires. Remplace le camembert par des barres horizontales triées, la plus grande en haut, et affiche la part de chaque produit en pourcentage au bout de sa barre (`ax.bar_label`), en prolongeant l'axe horizontal jusqu'à 75 000 euros. Donne-lui un titre qui énonce le message.

**Résultat attendu** :

![Résultat attendu de la question 4.5](attendus/q4_5.png)

In [ ]:
ca_produit = ventes.groupby("produit")["montant"].sum()

fig, ax = plt.subplots(figsize=(7, 7))
ax.pie(ca_produit.values, labels=ca_produit.index, autopct="%.0f%%")
ax.set_title("Répartition du chiffre d'affaires par produit")


In [ ]:
# Vérification
from matplotlib.patches import Rectangle

assert all(isinstance(barre, Rectangle) for barre in ax.patches), "remplace le camembert (ax.pie) par des barres horizontales (ax.barh)"
largeurs = [barre.get_width() for barre in ax.patches]
assert len(largeurs) == 17, "il faut une barre par produit"
assert largeurs == sorted(largeurs), "trie les barres, la plus grande en haut"
assert len(ax.texts) == 17, "affiche la part de chaque produit au bout de sa barre avec ax.bar_label"
print("Question 4.5 : OK")


_Ta réponse ici._

### Pour aller plus loin 4.6

Le graphique suivant superpose le nombre de commandes et le panier moyen de chaque mois, avec deux axes verticaux (`ax.twinx()`). Remplace-le par deux graphiques l'un au-dessus de l'autre, qui partagent l'axe des mois : `fig, axes = plt.subplots(2, 1, figsize=(8, 5), sharex=True)`. Donne un titre à chacun.

**Résultat attendu** :

![Résultat attendu de la question 4.6](attendus/q4_6.png)

In [ ]:
panier_mensuel = mensuel / commandes

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(etiquettes, commandes.values, color="#B8C2CC")
ax2 = ax.twinx()
ax2.plot(etiquettes, panier_mensuel.values, color="#1F3A5F", marker="o")
ax2.set_ylim(160, 220)
ax.set_title("Nombre de commandes et panier moyen")


In [ ]:
# Vérification
assert len(fig.axes) == 2, "il faut deux graphiques"
assert fig.axes[0].get_position().bounds != fig.axes[1].get_position().bounds, "les deux graphiques doivent être séparés, l'un au-dessus de l'autre, et non superposés"
print("Question 4.6 : OK")


_Ta réponse ici._

### Pour aller plus loin 4.7

La heatmap suivante utilise une palette rouge, jaune et vert. Remplace-la par une palette lisible par les daltoniens, toujours centrée sur 100 % : tu dois retrouver la heatmap de la question 2.7.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
sns.heatmap(taux, annot=True, fmt=".0%", cmap="RdYlGn", center=1, cbar=False, ax=ax)
ax.set_title("Taux de réalisation de l'objectif mensuel")


In [ ]:
# Vérification
assert "Gn" not in ax.collections[0].get_cmap().name, "choisis une palette sans vert, par exemple RdBu"
print("Question 4.7 : OK")


_Ta réponse ici._

## Partie 5 : tableau de bord

*À faire après la section 6 du cours : construire un tableau de bord.*

Construis 4 à 5 graphiques qui, ensemble, racontent l'activité commerciale de janvier à septembre 2025 à la direction. Tu peux reprendre et améliorer ceux des parties précédentes.

Pour chaque graphique, respecte la structure ci-dessous : la question à laquelle il répond, le graphique, et ce qu'il faut en retenir en une ou deux phrases. Les questions proposées sont celles que se pose la direction ; tu peux les remplacer par d'autres si tu les juges plus utiles.

Avant de finir, vérifie chaque graphique avec cette liste :

- le type de graphique correspond à la question posée ;
- le titre énonce le message ;
- les axes sont nommés, avec leur unité, et les nombres sont lisibles ;
- les barres sont triées et partent de zéro ;
- la couleur sert à mettre en évidence, pas à décorer ;
- la palette est lisible par les daltoniens.

### Graphique 1

**Question** : comment le chiffre d'affaires a-t-il évolué depuis janvier ?

In [ ]:
# TODO


**Ce qu'on retient** :

### Graphique 2

**Question** : quelles régions portent l'activité ?

In [ ]:
# TODO


**Ce qu'on retient** :

### Graphique 3

**Question** : les régions atteignent-elles leurs objectifs ?

In [ ]:
# TODO


**Ce qu'on retient** :

### Graphique 4

**Question** : comment chaque région a-t-elle évolué mois après mois ? C'est le graphique interactif principal : exporte-le en `ca_mensuel_regions.html`.

In [ ]:
# TODO


In [ ]:
# Vérification
assert Path("ca_mensuel_regions.html").exists(), "exporte le graphique interactif principal en ca_mensuel_regions.html"
print("Graphique 4 : OK")


**Ce qu'on retient** :

### Graphique 5 (facultatif)

**Question** : à toi de la choisir, par exemple : quelles catégories font le chiffre d'affaires ?

In [ ]:
# TODO


**Ce qu'on retient** :

## Avant de rendre

- Les fichiers `activite_mensuelle.png` et `ca_mensuel_regions.html` sont présents dans le dossier du TP, et le fichier HTML s'ouvre dans un navigateur en restant interactif.
- Toutes les cellules de vérification affichent `OK`.
- Chaque graphique du tableau de bord a sa question et son message.
- Le notebook s'exécute sans erreur de haut en bas : menu *Kernel*, puis *Restart Kernel and Run All Cells*.